In [0]:
df_select_columns = spark.read \
                .option("header", True) \
                .option("inferSchema", True) \
                .csv("/Volumes/rentsight/airbnb-infos/01-raw/listings.csv")

df_select_columns = df_select_columns.select("id", "neighbourhood", "latitude", "longitude", "room_type", "price", "minimum_nights", "number_of_reviews", "last_review", "reviews_per_month", "availability_365")

display(df_select_columns)

In [0]:
#Ajuste no Id
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window = Window.orderBy(F.lit(1))  
df_new_id = df_select_columns.withColumn("id", F.row_number().over(window))

display(df_new_id)


In [0]:
# Vendo valores por coluna

for col_name in df_new_id.columns:
    print(f"\n=== Coluna: {col_name} ===")
    df_new_id.groupBy(col_name).count().orderBy("count", ascending=False).show(truncate=False)


In [0]:
# Corrigindo Schema

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, DateType

df_safe = df_new_id \
    .withColumn("price", F.expr("try_cast(price AS DOUBLE)")) \
    .withColumn("minimum_nights", F.expr("try_cast(minimum_nights AS INT)")) \
    .withColumn("number_of_reviews", F.expr("try_cast(number_of_reviews AS INT)")) \
    .withColumn("last_review", F.expr("try_cast(last_review AS DATE)")) \
    .withColumn("reviews_per_month", F.expr("try_cast(reviews_per_month AS DOUBLE)")) \
    .withColumn("availability_365", F.expr("try_cast(availability_365 AS INT)")) \
    .withColumn("room_type", F.expr("try_cast(room_type AS STRING)")) \
    .withColumn("latitude", F.expr("try_cast(latitude AS DOUBLE)")) \
    .withColumn("longitude", F.expr("try_cast(longitude AS DOUBLE)"))
display(df_safe)

In [0]:
df_safe.write.mode("overwrite").option("header", True).csv("/Volumes/rentsight/airbnb-infos/02-silver/listings_silver_rj")